In [0]:
from pyspark.sql import functions as F

storage_account = "sanassign"

silver_base = (
    f"abfss://silver@{storage_account}.dfs.core.windows.net/sales-view"
)

gold_base = (
    f"abfss://gold@{storage_account}.dfs.core.windows.net/sales-view"
)

In [0]:
product_df = (
    spark.read
    .format("delta")
    .load(f"{silver_base}/product")
)

store_df = (
    spark.read
    .format("delta")
    .load(f"{silver_base}/store")
)

sales_df = (
    spark.read
    .format("delta")
    .load(f"{silver_base}/customer_sales")
)

In [0]:
product_df.printSchema()
store_df.printSchema()
sales_df.printSchema()

In [0]:
product_store_df = (
    product_df.alias("p")
    .join(
        store_df.alias("st"),
        F.col("p.store_id") == F.col("st.store_id"),
        "left"
    )
    .select(
        F.col("st.store_id").alias("store_id"),
        F.col("st.store_name").alias("store_name"),
        F.col("st.location").alias("location"),
        F.col("st.manager_name").alias("manager_name"),
        F.col("p.product_id").alias("product_id"),
        F.col("p.product_name").alias("product_name"),
        F.col("p.product_code").alias("product_code"),
        F.col("p.description").alias("description"),
        F.col("p.category_id").alias("category_id"),
        F.col("p.price").alias("price"),
        F.col("p.stock_quantity").alias("stock_quantity"),
        F.col("p.supplier_id").alias("supplier_id"),
        F.col("p.created_at").alias("product_created_at"),
        F.col("p.updated_at").alias("product_updated_at"),
        F.col("p.image_url").alias("image_url"),
        F.col("p.weight").alias("weight"),
        F.col("p.expiry_date").alias("expiry_date"),
        F.col("p.is_active").alias("is_active"),
        F.col("p.tax_rate").alias("tax_rate")
    )
)

In [0]:
gold_df = (
    sales_df.alias("s")
    .join(
        product_store_df.alias("ps"),
        F.col("s.product_id") == F.col("ps.product_id"),
        "left"
    )
    .select(
        F.col("s.order_date").alias("OrderDate"),
        F.col("s.category").alias("Category"),
        F.col("s.city").alias("City"),
        F.col("s.customer_id").alias("CustomerID"),
        F.col("s.order_id").alias("OrderID"),
        F.col("s.product_id").alias("Product_ID"),
        F.col("s.profit").alias("Profit"),
        F.col("s.region").alias("Region"),
        F.col("s.sales").alias("Sales"),
        F.col("s.segment").alias("Segment"),
        F.col("s.ship_date").alias("ShipDate"),
        F.col("s.ship_mode").alias("ShipMode"),
        F.col("ps.store_name").alias("store_name"),
        F.col("ps.location").alias("location"),
        F.col("ps.manager_name").alias("manager_name"),
        F.col("ps.product_name").alias("product_name"),
        F.col("ps.price").alias("price"),
        F.col("ps.stock_quantity").alias("stock_quantity"),
        F.col("ps.image_url").alias("image_url")
    )
)

In [0]:
gold_path = f"{gold_base}/StoreProductSalesAnalysis"

(
    gold_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(gold_path)
)